# Notebook 02 — Image Quality Check

Verify CLAHE effect, augmentations, and DataLoader output.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

from src.utils.paths import load_config
from src.data.transforms import apply_clahe, get_transforms
from src.data.dataset import ERCPDataset

cfg = load_config('../config.yaml')
CLASS_NAMES = cfg['data']['class_names']
print('Ready')

## 1. CLAHE effect on example images

In [ ]:
from src.data.eda import get_sample_paths
samples = get_sample_paths(cfg['data']['train_dir'], CLASS_NAMES, n=1)

fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(8, len(CLASS_NAMES)*3))
for row, cls in enumerate(CLASS_NAMES):
    paths = samples.get(cls, [])
    if not paths:
        continue
    img = Image.open(paths[0]).convert('RGB')
    img_clahe = apply_clahe(img)
    axes[row][0].imshow(img)
    axes[row][0].set_title(f'{cls}\nOriginal')
    axes[row][0].axis('off')
    axes[row][1].imshow(img_clahe)
    axes[row][1].set_title(f'{cls}\nCLAHE')
    axes[row][1].axis('off')
plt.suptitle('Original vs CLAHE', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Augmented images preview

In [ ]:
import torch

train_ds = ERCPDataset(cfg['data']['train_dir'],
                       transform=get_transforms(224, 'train'),
                       class_names=CLASS_NAMES)

# Inverse normalize for display
import torchvision.transforms as T
inv_norm = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

if len(train_ds) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    idx = 0
    for ax in axes.flat:
        img_tensor, label = train_ds[idx % len(train_ds)]
        img_disp = inv_norm(img_tensor).permute(1,2,0).numpy()
        img_disp = np.clip(img_disp, 0, 1)
        ax.imshow(img_disp)
        ax.set_title(CLASS_NAMES[label], fontsize=8)
        ax.axis('off')
        idx += 1
    plt.suptitle('Augmented Training Images (normalized then recovered)', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No training images found.')